# Scraper Strony Booking

In [ ]:
import csv
import json
import re
import asyncio
from pathlib import Path
from typing import Any, Dict, List
from urllib.parse import unquote, urlparse
from playwright.async_api import async_playwright, Error as PlaywrightError

In [ ]:



# KONFIGURACJA

TARGET_URL = "https://www.booking.com/hotel/pl/next-to-royal-castle.pl.html"  
ROUNDS = 20           
WAIT_MS = 1500        
MAX_PAGES = 20        
HEADED = True        
DEBUG = True          
AUTOSAVE_EVERY = 3    

DEFAULT_OPEN_REVIEWS_SELECTOR = "a[rel='reviews'][href='#blockdisplay4']"
DEFAULT_NEXT_SELECTOR = "button[aria-label='Następna strona']"


#  FUNKCJE POMOCNICZE I CZYSZCZĄCE

def normalize(text: str) -> str:
    return re.sub(r"\s+", " ", (text or "")).strip()

def slugify(text: str) -> str:
    s = normalize(text).lower().replace("+", " ")
    s = re.sub(r"[^a-z0-9ąćęłńóśźż]+", "-", s, flags=re.IGNORECASE).strip("-")
    return s or "obiekt"

def infer_booking_location_slug(url: str) -> str:
    try: path = unquote(urlparse(url).path or "")
    except Exception: path = ""
    m = re.search(r"/hotel/[^/]+/([^/]+?)(?:\\.html)?$", path)
    return slugify(m.group(1)) if m else "booking_obiekt"

def log(debug: bool, msg: str) -> None:
    if debug: print(f"  [DEBUG] {msg}")

def parse_rating(raw: str) -> float | None:
    t = normalize(raw)
    if not t: return None
    m = re.search(r"([0-9]{1,2}(?:[.,][0-9])?)", t)
    if not m: return None
    try: v = float(m.group(1).replace(",", "."))
    except ValueError: return None
    if 0 <= v <= 10 or 0 <= v <= 5: return v
    return None

def dedupe(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen, out = set(), []
    for r in rows:
        key = (normalize(r.get("author", "")).lower(), normalize(r.get("date", "")).lower(), normalize(r.get("text", "")).lower())
        if key not in seen:
            seen.add(key)
            out.append(r)
    return out

def is_host_reply(text: str, author: str) -> bool:
    joined = f"{normalize(author)} {normalize(text)}".lower()
    patterns = (r"response from property", r"odpowiedz obiektu", r"owner response", r"host response", r"managed by", r"dziekujemy", r"zapraszamy ponownie", r"thank you for your review")
    return any(re.search(p, joined) for p in patterns)

def looks_like_noise(text: str) -> bool:
    t = normalize(text).lower()
    if len(t) < 5: return True
    noise = ("book now", "show on map", "availability", "facilities", "policies", "house rules", "check-in", "check-out", "property description", "wyswietlono", "wyświetlono", "nastepna strona", "następna strona", "poprzednia strona", "pomocna", "niezbyt pomocna", "helpful")
    return any(x in t for x in noise)

def clean_rows(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    out = []
    for r in rows:
        author, date, text = normalize(str(r.get("author", ""))), normalize(str(r.get("date", ""))), normalize(str(r.get("text", "")))
        if looks_like_noise(text) or is_host_reply(text, author) or len(text) > 1400: continue
        out.append({"author": author, "rating": r.get("rating"), "date": date, "text": text, "source_url": r.get("source_url", "")})
    return dedupe(out)


#  ASYNCHRONICZNE AKCJE PRZEGLĄDARKI

async def extract_reviews(page) -> List[Dict[str, Any]]:
    raw = await page.evaluate("""
        () => {
          const norm = (s) => (s || '').replace(/\\s+/g, ' ').trim();
          const pick = (root, selectors) => {
            for (const sel of selectors) {
              const el = root.querySelector(sel);
              if (el) { const txt = norm(el.innerText || el.textContent || ''); if (txt) return txt; }
            }
            return '';
          };
          const candidateSelectors = ["[data-testid*='review-card']", "[data-testid*='review_item']", "[data-testid*='review-list'] [data-testid*='card']", "[class*='review_list'] [class*='review']", "[class*='reviewlist'] [class*='review']", "[class*='review_item']", "[class*='review-item']", "article[class*='review']"];
          const seen = new Set(); const cards = []; const cardTexts = new Set();
          
          for (const sel of candidateSelectors) {
            for (const node of document.querySelectorAll(sel)) {
              if (!(node instanceof HTMLElement)) continue;
              const txt = norm(node.innerText || '');
              if (txt.length < 80 || txt.length > 2200) continue;
              if (!/review|opinia|opinie|guest|gosc|go[śs]ci|ocena|score|stayed|pobyt|ocena dodana/i.test(txt)) continue;
              if (!/(pomocna|helpful|ocena dodana|reviewed|stayed|pobyt)/i.test(txt)) continue;
              if (seen.has(node)) continue;
              if (cardTexts.has(txt)) continue;
              seen.add(node); cardTexts.add(txt); cards.push(node);
            }
          }
          if (cards.length === 0) {
            const scoreNodes = Array.from(document.querySelectorAll("div, span")).filter((el) => /^(10|[0-9](?:[.,][0-9])?)$/.test(norm(el.innerText || '')));
            for (const sn of scoreNodes) {
              let cur = sn;
              for (let depth = 0; depth < 8 && cur; depth++) {
                cur = cur.parentElement;
                if (!cur) break;
                const txt = norm(cur.innerText || '');
                if (txt.length < 80 || txt.length > 2200) continue;
                if (!/(ocena dodana|reviewed|stayed|pobyt)/i.test(txt)) continue;
                if (cardTexts.has(txt)) break;
                cardTexts.add(txt); cards.push(cur); break;
              }
            }
          }
          const rows = [];
          for (const c of cards) {
            const author = pick(c, ["[data-testid*='review-card-name']", "[data-testid*='reviewer-name']", "[class*='reviewer']", "[class*='avatar'] [class*='title']", "strong"]);
            const date = pick(c, ["time", "[data-testid*='review-date']", "[class*='reviewDate']", "[class*='review-date']"]);
            const ratingRaw = pick(c, ["[data-testid*='review-score']", "[data-testid*='score']", "[aria-label*='Scored' i]", "[aria-label*='Rated' i]", "[class*='review-score']", "[class*='score']"]);
            const bodyParts = [];
            for (const sel of ["[data-testid*='review-positive-text']", "[data-testid*='review-negative-text']", "[data-testid*='review-comment']", "[class*='review-pos']", "[class*='review-neg']", "[class*='review-text']", "[class*='review-body']", "p"]) {
              for (const el of c.querySelectorAll(sel)) {
                const txt = norm(el.innerText || el.textContent || '');
                if (txt && txt.length >= 8) bodyParts.push(txt);
              }
            }
            let text = norm(bodyParts.join(' '));
            if (!text) text = norm(c.innerText || '');
            if (text.length < 8 || text.length > 1400) continue;
            rows.push({ author, date, rating_raw: ratingRaw, text, source_url: window.location.href });
          }
          return rows;
        }
    """)
    out = [{"author": normalize(str(r.get("author", ""))), "date": normalize(str(r.get("date", ""))), "rating": parse_rating(str(r.get("rating_raw", ""))), "text": normalize(str(r.get("text", ""))), "source_url": str(r.get("source_url", ""))} for r in raw]
    return clean_rows(out)

async def click_cookie_accept(page) -> None:
    patterns = (r"accept", r"zaakceptuj", r"zgadzam", r"allow all", r"akceptuj")
    for pat in patterns:
        for role in ("button", "link"):
            loc = page.get_by_role(role, name=re.compile(pat, re.IGNORECASE))
            if await loc.count() > 0:
                try:
                    await loc.first.click(timeout=4000)
                    await page.wait_for_timeout(1500)
                    return
                except Exception: pass

async def open_reviews_area(page, force_selector: str = "", debug: bool = False) -> None:
    if force_selector:
        try:
            loc = page.locator(force_selector)
            if await loc.count() > 0:
                await loc.first.click(timeout=2500)
                await page.wait_for_timeout(1200)
                log(debug, f"Otwarto opinie wymuszonym selektorem: {force_selector}")
                return
        except Exception: pass
    patterns = (r"guest reviews", r"reviews", r"opinie gosci", r"opinie", r"see all reviews", r"zobacz wszystkie opinie")
    for pat in patterns:
        for role in ("button", "link"):
            loc = page.get_by_role(role, name=re.compile(pat, re.IGNORECASE))
            if await loc.count() > 0:
                try:
                    await loc.first.click(timeout=2000)
                    await page.wait_for_timeout(1000)
                    log(debug, f"Otwarto opinie na podstawie tekstu: {pat}")
                    return
                except Exception: pass

async def click_expand_buttons(page) -> None:
    for pat in (r"read more", r"more", r"pokaz wiecej", r"zobacz wiecej"):
        loc = page.get_by_role("button", name=re.compile(pat, re.IGNORECASE))
        count = await loc.count()
        for i in range(min(count, 30)):
            try: await loc.nth(i).click(timeout=300)
            except Exception: pass

async def click_next_page(page, force_selector: str = "", debug: bool = False) -> bool:
    if force_selector:
        try:
            loc = page.locator(force_selector)
            if await loc.count() > 0 and await loc.first.get_attribute("disabled") is None:
                await loc.first.click(timeout=1800)
                await page.wait_for_load_state("networkidle", timeout=8000)
                await page.wait_for_timeout(1200)
                log(debug, "Następna strona (wymuszony selektor).")
                return True
        except Exception: pass
    selectors = ["a[rel='next']", "button[aria-label*='Next' i]", "button[aria-label*='Następna' i]", "button[aria-label*='Nastepna' i]", "button[aria-label*='Dalej' i]", "[data-testid*='pagination-next']"]
    for sel in selectors:
        loc = page.locator(sel)
        if await loc.count() > 0:
            try:
                if await loc.first.get_attribute("disabled") is None:
                    await loc.first.click(timeout=1800)
                    await page.wait_for_timeout(1200)
                    log(debug, f"Następna strona (selektor {sel}).")
                    return True
            except Exception: pass
    for pat in (r"next", r"nast[eę]pna", r"dalej"):
        for role in ("button", "link"):
            loc = page.get_by_role(role, name=re.compile(pat, re.IGNORECASE))
            if await loc.count() > 0:
                try:
                    await loc.first.click(timeout=1800)
                    await page.wait_for_timeout(1200)
                    return True
                except Exception: pass
    return False

async def scroll_once(page) -> None:
    await page.evaluate("""() => { const h = Math.max(window.innerHeight || 1200, 1200); window.scrollBy(0, Math.floor(h * 0.9)); const scrollers = Array.from(document.querySelectorAll('div')).filter((el) => { const cs = window.getComputedStyle(el); return (cs.overflowY.includes('auto') || cs.overflowY.includes('scroll')) && el.scrollHeight > el.clientHeight + 50; }); for (const el of scrollers) { el.scrollTop = Math.min(el.scrollTop + Math.max(1000, el.clientHeight * 1.5), el.scrollHeight); } }""")

def save_json(path: Path, rows: List[Dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")

def save_csv(path: Path, rows: List[Dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["author", "rating", "date", "text", "source_url"])
        writer.writeheader()
        writer.writerows(rows)

## Glówna pętła

In [ ]:


async def run_scraper():
    location_slug = infer_booking_location_slug(TARGET_URL)
    out_json = Path(f"{location_slug}.json")
    out_csv = Path(f"{location_slug}.csv")

    print(f" Uruchamiam pobieranie dla: {location_slug}")

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=not HEADED)
        page = await browser.new_page(locale="pl-PL")
        
        print(" Wczytywanie strony...")
        await page.goto(TARGET_URL, wait_until="domcontentloaded", timeout=120000)
        await page.wait_for_timeout(2500)

        await click_cookie_accept(page)
        await open_reviews_area(page, force_selector=DEFAULT_OPEN_REVIEWS_SELECTOR, debug=DEBUG)

        all_rows = []
        no_growth_rounds = 0
        visited_pages = 1

        try:
            for i in range(max(1, ROUNDS)):
                await click_expand_buttons(page)
                before = len(all_rows)
                
                # Ekstrakcja asynchroniczna
                extracted = await extract_reviews(page)
                all_rows.extend(extracted)
                all_rows = dedupe(all_rows)
                after = len(all_rows)
                
                print(f"[Runda {i+1}/{ROUNDS}] Znaleziono unikalnych opinii: {after} | Odwiedzone strony: {visited_pages}")

                if (i + 1) % max(1, AUTOSAVE_EVERY) == 0:
                    save_json(out_json, all_rows)
                    save_csv(out_csv, all_rows)

                no_growth_rounds = no_growth_rounds + 1 if after == before else 0

                moved = False
                if visited_pages < MAX_PAGES:
                    moved = await click_next_page(page, force_selector=DEFAULT_NEXT_SELECTOR, debug=DEBUG)
                    if moved:
                        visited_pages += 1

                if not moved:
                    await scroll_once(page)
                    await page.mouse.wheel(0, 3200)

                await page.wait_for_timeout(WAIT_MS)
                if no_growth_rounds >= 20:
                    print("Brak nowych opinii przez wiele iteracji.")
                    break

        except KeyboardInterrupt:
            print("\n Przerwano. Zapisano stan częściowy.")
        except PlaywrightError:
            try: await page.wait_for_load_state("domcontentloaded", timeout=7000)
            except PlaywrightError: pass

        # Zamknięcie
        final_extract = await extract_reviews(page)
        all_rows.extend(final_extract)
        rows = dedupe(all_rows)
        save_json(out_json, rows)
        await browser.close()

    print(f"\n Zakończono! Pomyślnie zapisano {len(rows)} opinii.")
    print(f"Pliki zapisano jako: {out_json} oraz {out_csv}")

# Wywołanie w Jupyterze wymaga await
await run_scraper()